Importo las librerías para trabajar

In [1]:
import pandas as pd
import requests
import shutil
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
from scipy import sparse
from surprise import Dataset, Reader
from surprise import SVD
from surprise import accuracy
from surprise.model_selection import train_test_split
import gradio as gr

In [2]:
dfml=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/ML_ETL_plataformas.csv')

In [3]:
dfmlr1=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/1.csv')
dfmlr2=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/2.csv')
dfmlr3=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/3.csv')
dfmlr4=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/4.csv')
dfmlr5=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/5.csv')
dfmlr6=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/6.csv')
dfmlr7=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/7.csv')
dfmlr8=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/8.csv')

In [4]:
dfmlr=(pd.concat([dfmlr1,dfmlr2,dfmlr3,dfmlr4,dfmlr5,dfmlr6,dfmlr7,dfmlr8], axis=0))

Elimino la columna 'timespan' de df_users porque no la vamos a utilizar

In [5]:
dfmlr = dfmlr.drop(['timestamp'], axis=1)

In [6]:
dfml.head(1)

,id,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,plataforma,duration_int,duration_type,score
0,as1,s1,movie,the grand seduction,don mckellar,"brendan gleeson, taylor kitsch, gordon pinsent",canada,2021-03-30 00:00:00,2014,G,113 min,"comedy, drama",a small fishing village must procure a local d...,amazon,113,min,3.549127


Elimino columna no relevantes para el análisis

In [7]:
dfml= dfml.drop(['show_id','director','cast','country','date_added','release_year','rating','duration','description','duration_int','duration_type','score'], axis=1)

In [8]:
dfml.head(1)

,id,type,title,listed_in,plataforma
0,as1,movie,the grand seduction,"comedy, drama",amazon


Cambio de nombre de la columna 'id' a 'movieId'

In [9]:
dfml = dfml.rename(columns={'id': 'movieId'})
# We have now the same colum 'movieId' for both dfs

In [10]:
dfml.head(2)

,movieId,type,title,listed_in,plataforma
0,as1,movie,the grand seduction,"comedy, drama",amazon
1,as2,movie,take care good night,"drama, international",amazon


Uno los dos df

In [21]:
dfcompl = pd.merge(left=dfmlr,right=dfml)

In [11]:
dfcompl.shape
# we have 11024289 rows 

NameError: name 'dfcompl' is not defined

In [26]:
dfcompl.head(5)

,userId,rating,movieId,type,title,listed_in,plataforma
0,1,1.0,as680,tv show,the english civil war,"documentary, special interest",amazon
1,583,4.5,as680,tv show,the english civil war,"documentary, special interest",amazon
2,765,5.0,as680,tv show,the english civil war,"documentary, special interest",amazon
3,2116,3.0,as680,tv show,the english civil war,"documentary, special interest",amazon
4,2143,3.0,as680,tv show,the english civil war,"documentary, special interest",amazon


Reordenar mi variable de objetivos

In [29]:
columnas = dfcompl.columns.tolist()
columnas = ['userId', 'movieId', 'type', 'title', 'listed_in', 'plataforma', 'rating']
dfcompl = dfcompl[columnas]
dfcompl.head(2)

,userId,movieId,type,title,listed_in,plataforma,rating
0,1,as680,tv show,the english civil war,"documentary, special interest",amazon,1.0
1,583,as680,tv show,the english civil war,"documentary, special interest",amazon,4.5


Realizo una copia de dfcompl como mejor práctica

In [30]:
df_ml = dfcompl.copy()
df_ml.sample(3)

Creación de una lista de valores de la columna "listed_in
- Esta parte está comentada en esta guía

In [ ]:
# df_ml['listed_in'] = df_ml.listed_in.str.split(' , ')

Rango de verificación de los valores de "rating

In [31]:
print(df_ml['rating'].min())
print(df_ml['rating'].max())
# rating values goes between of 0.5 - 5

0.5
5.0


Preparando mi df con la columna 'userId' 'movieId' 'rating'

In [32]:
df_prepared = df_ml.loc[:,["userId","movieId","rating"]]

Muestreo de 1000000 filas

In [33]:
df_muest = df_prepared.sample(n=1000000, replace=True)

In [34]:
df_muest.head()

,userId,movieId,rating
9362722,69738,as4401,3.5
6452135,53268,as4256,3.0
10526316,37461,as228,3.0
2251904,269583,ns4724,3.5
10488530,71590,ns2976,4.0


# Modelando

Codificación de la columna movieId a int

In [39]:
col = df_muest.movieId

# creamos objeto labelencoder y ajustamos la columna
labelE = LabelEncoder()
labelE.fit(col)

#tranformamos la columna
col_transformada = labelE.transform(col)

#lo reemplazamos a show id original
df_muest.movieId = col_transformada
df_muest.head()

,userId,movieId,rating
9362722,69738,3781,3.5
6452135,53268,3619,3.0
10526316,37461,1423,3.0
2251904,269583,18330,3.5
10488530,71590,16387,4.0


Codificación de la columna movieid a int

In [ ]:
# df_prepared = df_ml.loc[:,["userId","movieId","rating"]]

In [ ]:
# df_prepared.head()

In [ ]:
# from scipy.sparse import csr_matrix

# sparse_matrix = csr_matrix(df_prepared.values)

In [41]:
# Convert dataframes to surprise datasets
reader = Reader(line_format='user item rating', rating_scale=(0.5, 5))
data = Dataset.load_from_df(df_muest, reader)

In [42]:
# Build full trainset
data_train_surp = data.build_full_trainset()

# Define the model
svd = SVD()

# Train the model
svd.fit(data_train_surp)

In [43]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [44]:
predictions = svd.test(testset)

In [45]:
predictions

[Prediction(uid=43558, iid=20574, r_ui=4.0, est=3.2190426559891807, details={'was_impossible': False}),
 Prediction(uid=70364, iid=16080, r_ui=5.0, est=3.6521908680346096, details={'was_impossible': False}),
 Prediction(uid=53352, iid=1307, r_ui=5.0, est=3.83866052331602, details={'was_impossible': False}),
 Prediction(uid=120169, iid=21768, r_ui=5.0, est=4.446494722601759, details={'was_impossible': False}),
 Prediction(uid=74520, iid=4437, r_ui=5.0, est=4.229485098887742, details={'was_impossible': False}),
 Prediction(uid=264720, iid=12773, r_ui=4.0, est=3.6056433147188387, details={'was_impossible': False}),
 Prediction(uid=113568, iid=10190, r_ui=4.0, est=3.8686060809198093, details={'was_impossible': False}),
 Prediction(uid=109775, iid=12222, r_ui=3.0, est=3.4471356425320656, details={'was_impossible': False}),
 Prediction(uid=262210, iid=9701, r_ui=3.0, est=3.1489246935546764, details={'was_impossible': False}),
 Prediction(uid=118292, iid=8616, r_ui=4.0, est=3.990261147202036,

In [46]:
svd.predict(356,567,4)

Prediction(uid=356, iid=567, r_ui=4, est=3.729538794417126, details={'was_impossible': False})